# Week 2, Day 4: how good is the RAG pipeline? (evaluation, step by step)

A RAG pipeline has three stages: **retrieve** the right text, **gate** (refuse when the evidence is weak), **generate** an answer with a citation.
If an answer is wrong you must know **which stage failed**. This notebook scores retrieval and answers separately, labels every failure, and shows which change helps.

It uses the **real files** of the project (`vector-lab`): the 12 documents, the 20 evaluation questions, and the scoring code from `rag.py`.
**Simplified here:** the database is a list in memory, and keyword search is word counting, so numbers can differ slightly from the project's evidence page.
**Important:** the answer step is *extractive* (it picks one sentence of the retrieved text and cites it). No language model is used, so this measures retrieval, the filter and the gate, not how a model writes.

Run the cells from the top, one at a time (Shift+Enter).

## Step 0: install the tools (about 1 minute)

In [ ]:
!pip install -q fastembed langchain-text-splitters numpy

## Step 1: the documents and the evaluation set
Two real files from the project. `corpus.py` has 12 invented company documents. `rag_eval_set.py` has **20 questions**, each with the expected answer, the source document, the exact fact a correct answer must contain, and the profile filter of the user (region, year). Four questions are **not in the documents** on purpose.

In [ ]:
%%writefile corpus.py
"""The approved document set (an invented company handbook) and the fixed question set for Week 2, Day 3.

Unlike the 4-document sample used on Days 1 and 2, this set has near-duplicate policies that differ by region and year
(annual leave in the UK and in Germany, an archived 2024 leave policy, hotel caps in Europe and the US). Without a metadata
filter those questions have no single right answer, which is the point of Day 3.
"""
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True)
class Doc:
    doc_id: str
    title: str
    department: str
    region: str  # "All", "UK", "Germany", "US"
    year: int
    body: str


@dataclass(frozen=True)
class Q:
    question: str
    kind: str  # plain | paraphrase | filtered | out_of_scope
    docs: tuple[str, ...] = ()  # doc_ids that count as a correct source; empty = not in the documents
    phrase: str | None = None  # text a correct passage must contain
    filter: dict = field(default_factory=dict)  # metadata filter that a careful user would apply


DOCS: list[Doc] = [
    Doc("leave-uk-2025", "Leave policy (UK)", "hr", "UK", 2025,
        "Every full-time UK employee gets 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per month. "
        "Up to 10 unused days can be carried over to the next year; anything above that expires on 31 March.\n\n"
        "Sick leave is separate from annual leave: 12 days per year. A medical certificate is needed when sick leave lasts more than 2 consecutive days. "
        "Tell your manager before 10:00 on the first day of absence.\n\n"
        "Parental leave is 26 weeks for the primary caregiver and 4 weeks for the other parent. Requests must be sent to HR at least 8 weeks before the expected start date."),
    Doc("leave-uk-2024", "Leave policy (UK, archived 2024)", "hr", "UK", 2024,
        "This policy is archived and was replaced in 2025. UK employees received 20 days of paid annual leave per calendar year. "
        "Up to 5 unused days could be carried over and the rest expired on 31 January.\n\n"
        "Sick leave was 10 days per year and a medical certificate was needed after 3 consecutive days of absence."),
    Doc("leave-de-2025", "Leave policy (Germany)", "hr", "Germany", 2025,
        "Every full-time employee in Germany gets 30 days of paid annual leave per calendar year, in line with the works agreement. "
        "Up to 5 unused days can be carried over; anything above that expires on 31 March.\n\n"
        "Sick leave is paid in full for up to 6 weeks. A medical certificate is needed from the fourth day of absence. Tell your manager before 09:00 on the first day.\n\n"
        "Public holidays follow the federal state of the office. Bridge days are announced by the people team in January."),
    Doc("expense-all-2025", "Expense policy", "finance", "All", 2025,
        "Business expenses are reimbursed within 14 days of a complete claim. Submit claims in the finance portal with the original receipt attached. Claims without receipts are rejected.\n\n"
        "Flights must be booked in economy class at least 14 days ahead. Hotel stays are capped at 120 euros per night in most cities and 180 euros in London. Meals during travel are covered up to 45 euros per day.\n\n"
        "Any single purchase above 500 euros needs written approval from a director before you buy. Software subscriptions are bought only by the IT team, never on a personal card."),
    Doc("travel-us-2025", "Travel policy (United States)", "finance", "US", 2025,
        "Employees travelling within the United States book flights in economy class at least 21 days ahead. Hotel stays are capped at 250 dollars per night in New York and San Francisco and 170 dollars elsewhere.\n\n"
        "Meals during travel are covered up to 75 dollars per day. Rental cars need approval from the team lead and must be the smallest suitable class. Claims are paid within 30 days."),
    Doc("remote-all-2025", "Remote work and security", "it", "All", 2025,
        "Employees may work remotely up to 3 days a week after completing their probation period of 3 months. Team days in the office are Tuesday and Thursday.\n\n"
        "Company laptops must use full disk encryption and lock automatically after 5 minutes. Use the VPN on any public Wi-Fi. Never store customer data on personal devices or in personal cloud drives.\n\n"
        "Report a lost device or a suspected phishing email to the security desk within 1 hour. Passwords must have at least 14 characters and multi-factor authentication is mandatory for all company accounts."),
    Doc("remote-de-2025", "Remote work addendum (Germany)", "it", "Germany", 2025,
        "Under the works agreement employees in Germany may work remotely up to 2 days a week. A home-office allowance of 40 euros per month is paid with the salary.\n\n"
        "Working time is recorded in the time-tracking tool and rest periods of 11 hours between working days are respected. Equipment such as a monitor can be ordered once through IT."),
    Doc("onboarding-all-2025", "Onboarding and training", "hr", "All", 2025,
        "New starters get a buddy for the first 8 weeks. The first week covers tools, security training and a tour of the main systems. A check-in with the manager happens at 30, 60 and 90 days.\n\n"
        "Each employee has a learning budget of 1,500 euros per year for courses, books and conferences. Conference attendance needs manager approval at least 30 days before the event. The learning budget does not roll over.\n\n"
        "Mentoring is open to everyone: ask the people team to match you with a mentor from another department. Mentoring pairs meet once every 2 weeks for 45 minutes."),
    Doc("incident-all-2025", "Security incident response", "it", "All", 2025,
        "A security incident is any lost device, leaked credential, suspected phishing email or unauthorised access. Report it to the security desk within 1 hour of noticing it, by phone or through the incident form.\n\n"
        "The security desk assigns a severity from 1 to 4 and a responder. Severity 1 incidents are reviewed by the leadership team within 24 hours. A written summary is shared within 5 working days."),
    Doc("procurement-all-2025", "Procurement and vendors", "finance", "All", 2025,
        "Vendor contracts above 5,000 euros need 3 written quotes and approval from the finance director. Contracts above 50,000 euros also need a security review and legal sign-off.\n\n"
        "Preferred vendors are listed in the finance portal. New vendors must complete the supplier questionnaire before the first purchase order is issued."),
    Doc("performance-all-2025", "Performance reviews", "hr", "All", 2025,
        "Performance reviews take place twice a year, in June and in December. Each review combines a self-assessment, manager feedback and a rating from 1 to 5.\n\n"
        "Pay changes are decided once a year after the December review. An employee who disagrees with a rating can ask for a second review by the next-level manager within 10 working days."),
    Doc("holidays-uk-2026", "Office holidays 2026 (UK)", "hr", "UK", 2026,
        "UK offices are closed on 1 January, 3 April, 6 April, 4 May, 25 May, 31 August, 25 December and 28 December 2026. The office closes at 14:00 on 24 December.\n\n"
        "Staff who work on a bank holiday receive a day off in lieu within 3 months."),
]

QUESTIONS: list[Q] = [
    # Plain: the question uses the words of one document and has one right source.
    Q("How soon are expense claims reimbursed?", "plain", ("expense-all-2025",), "within 14 days"),
    Q("After how many minutes does a company laptop lock itself?", "plain", ("remote-all-2025",), "5 minutes"),
    Q("How long must a password be?", "plain", ("remote-all-2025",), "14 characters"),
    Q("What is the yearly learning budget?", "plain", ("onboarding-all-2025",), "1,500 euros"),
    Q("How often do mentoring pairs meet?", "plain", ("onboarding-all-2025",), "once every 2 weeks"),
    Q("How many written quotes does a vendor contract above 5,000 euros need?", "plain", ("procurement-all-2025",), "3 written quotes"),
    Q("How often are performance reviews held?", "plain", ("performance-all-2025",), "twice a year"),
    Q("How long does a new starter have a buddy?", "plain", ("onboarding-all-2025",), "8 weeks"),
    Q("Within how long must a phishing email be reported?", "plain", ("remote-all-2025", "incident-all-2025"), "within 1 hour"),
    Q("How is the severity of a security incident rated?", "plain", ("incident-all-2025",), "severity from 1 to 4"),
    # Paraphrase: other words for the same fact (keyword search is expected to struggle).
    Q("Can I work from home?", "paraphrase", ("remote-all-2025", "remote-de-2025"), "remotely"),
    Q("Who signs off on a purchase above 500 euros?", "paraphrase", ("expense-all-2025",), "director"),
    Q("What do I need to do before buying something expensive?", "paraphrase", ("expense-all-2025", "procurement-all-2025"), "approval"),
    Q("When can I appeal my performance score?", "paraphrase", ("performance-all-2025",), "second review"),
    Q("What does the company pay for my courses and conferences?", "paraphrase", ("onboarding-all-2025",), "learning budget"),
    Q("Which days are people expected to be in the office?", "paraphrase", ("remote-all-2025",), "Tuesday and Thursday"),
    # Filtered: near-duplicate documents; the filter picks the right one. Run with and without it.
    Q("How many days of paid annual leave do I get?", "filtered", ("leave-uk-2025",), "24 days", {"$and": [{"region": "UK"}, {"year": {"$gte": 2025}}]}),
    Q("How many days of paid annual leave do I get?", "filtered", ("leave-de-2025",), "30 days", {"region": "Germany"}),
    Q("What is the hotel cap per night?", "filtered", ("travel-us-2025",), "250 dollars", {"region": "US"}),
    Q("How many days a week can I work remotely?", "filtered", ("remote-de-2025",), "2 days a week", {"region": "Germany"}),
    Q("How many unused leave days can be carried over?", "filtered", ("leave-uk-2025",), "10 unused days", {"$and": [{"region": "UK"}, {"year": {"$gte": 2025}}]}),
    Q("When does a medical certificate become necessary?", "filtered", ("leave-de-2025",), "fourth day", {"region": "Germany"}),
    # Not in the documents: the assistant must say there is not enough evidence.
    Q("What is the company dress code?", "out_of_scope"),
    Q("How much is the annual bonus?", "out_of_scope"),
    Q("When do stock options vest?", "out_of_scope"),
]

In [ ]:
%%writefile rag_eval_set.py
"""Week 2, Day 4: the RAG evaluation set. 20 questions with the expected answer, the source documents that hold it, the exact
fact (`phrase`) a correct answer must contain, and the profile filter a signed-in user would have (region and year).

Same 12 documents as Day 3 (app/corpus.py). 16 questions have an answer in the documents, 4 do not. Some are hard on purpose:
near-duplicate policies (UK, Germany, archived 2024), a fact that only appears in one of several documents, and a question
about a topic that is close to a real one but not covered (the answer must be a refusal, not a guess).
"""
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True)
class RQ:
    question: str
    expected: str  # the expected answer in plain words
    docs: tuple[str, ...]  # doc_ids that hold the answer; empty = not in the documents
    phrase: str | None  # exact text a correct answer (and its source chunk) must contain
    kind: str  # plain | paraphrase | filtered | multi | out_of_scope
    filter: dict = field(default_factory=dict)  # the user's profile filter; used only when the pipeline applies filters


UK25 = {"$and": [{"region": "UK"}, {"year": {"$gte": 2025}}]}
DE = {"region": "Germany"}
US = {"region": "US"}

RAG_QUESTIONS: list[RQ] = [
    # plain: the question uses the document's own words
    RQ("How soon are expense claims reimbursed?", "Within 14 days of a complete claim.", ("expense-all-2025",), "within 14 days", "plain"),
    RQ("After how many minutes does a company laptop lock itself?", "After 5 minutes.", ("remote-all-2025",), "5 minutes", "plain"),
    RQ("How long must a password be?", "At least 14 characters.", ("remote-all-2025",), "14 characters", "plain"),
    RQ("What is the yearly learning budget?", "1,500 euros per year.", ("onboarding-all-2025",), "1,500 euros", "plain"),
    RQ("How often are performance reviews held?", "Twice a year, in June and December.", ("performance-all-2025",), "twice a year", "plain"),
    RQ("How many written quotes does a vendor contract above 5,000 euros need?", "Three written quotes.", ("procurement-all-2025",), "3 written quotes", "plain"),
    # paraphrase: other words for the same fact
    RQ("Can I work from home?", "Yes, up to 3 days a week after probation.", ("remote-all-2025", "remote-de-2025"), "remotely", "paraphrase"),
    RQ("Who signs off on a purchase above 500 euros?", "A director, in writing, before you buy.", ("expense-all-2025",), "director", "paraphrase"),
    RQ("When can I appeal my performance score?", "Within 10 working days, to the next-level manager.", ("performance-all-2025",), "within 10 working days", "paraphrase"),
    RQ("Which days are people expected to be in the office?", "Tuesday and Thursday.", ("remote-all-2025",), "Tuesday and Thursday", "paraphrase"),
    # filtered: near-duplicate policies; the user's region and year pick the right one
    RQ("How many days of paid annual leave do I get?", "24 days (UK, 2025 policy).", ("leave-uk-2025",), "24 days", "filtered", UK25),
    RQ("How many days of paid annual leave do I get?", "30 days (Germany).", ("leave-de-2025",), "30 days", "filtered", DE),
    RQ("What is the hotel cap per night?", "250 dollars in New York and San Francisco, 170 dollars elsewhere (US).", ("travel-us-2025",), "250 dollars", "filtered", US),
    RQ("How many days a week can I work remotely?", "Up to 2 days a week (Germany).", ("remote-de-2025",), "2 days a week", "filtered", DE),
    # multi: the fact sits in one of two similar documents, the filter and the wording both matter
    RQ("How many unused leave days can be carried over?", "10 days (UK, 2025 policy).", ("leave-uk-2025",), "10 unused days", "multi", UK25),
    RQ("When does a medical certificate become necessary?", "From the fourth day of absence (Germany).", ("leave-de-2025",), "fourth day", "multi", DE),
    # out of scope: not in the documents; the correct behaviour is a refusal
    RQ("What is the company dress code?", "Not in the documents.", (), None, "out_of_scope"),
    RQ("How much is the annual bonus?", "Not in the documents.", (), None, "out_of_scope"),
    RQ("When do stock options vest?", "Not in the documents.", (), None, "out_of_scope"),
    RQ("How many days of paid maternity leave do I get in the US?", "Not in the documents (the US has no leave policy here).", (), None, "out_of_scope", US),
]

In [ ]:
from corpus import DOCS
from rag_eval_set import RAG_QUESTIONS
from collections import Counter
print(len(DOCS), "documents,", len(RAG_QUESTIONS), "questions")
print(Counter(q.kind for q in RAG_QUESTIONS))
q = RAG_QUESTIONS[10]
print()
print("question :", q.question)
print("expected :", q.expected)
print("source   :", q.docs, "| must contain:", repr(q.phrase))
print("user filter:", q.filter)

## Step 2: the search tools (same ideas as Day 3)
Cut the documents into chunks, turn them into numbers, search by meaning, by keywords, or both (hybrid). This is the Day 3 code in one cell. See the Day 3 notebook for the steps one by one.

In [ ]:
%%writefile engine_parts.py
from __future__ import annotations
def matches(meta: dict, flt: dict | None) -> bool:
    """Python check of a filter ($and, $or, $eq, $ne, $gt, $gte, $lt, $lte, $in). Used by the tests to verify what the database returned."""
    if not flt:
        return True
    for key, cond in flt.items():
        if key == "$and":
            if not all(matches(meta, c) for c in cond):
                return False
        elif key == "$or":
            if not any(matches(meta, c) for c in cond):
                return False
        else:
            v = meta.get(key)
            if not isinstance(cond, dict):
                cond = {"$eq": cond}
            for op, want in cond.items():
                ok = {"$eq": lambda: v == want, "$ne": lambda: v != want, "$gt": lambda: v is not None and v > want,
                      "$gte": lambda: v is not None and v >= want, "$lt": lambda: v is not None and v < want,
                      "$lte": lambda: v is not None and v <= want, "$in": lambda: v in want}.get(op)
                if ok is None:
                    raise ValueError(f"Unknown filter operator {op}")
                if not ok():
                    return False
    return True

In [ ]:
import re, numpy as np
from fastembed import TextEmbedding
from langchain_text_splitters import RecursiveCharacterTextSplitter
from engine_parts import matches

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = []
for d in DOCS:
    for i, text in enumerate(splitter.split_text(d.body)):
        chunks.append({"id": f"{d.doc_id}:{i+1}", "doc_id": d.doc_id, "title": d.title, "text": text,
                       "meta": {"department": d.department, "region": d.region, "year": d.year}})
model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
vectors = np.array(list(model.embed([c["text"] for c in chunks])), dtype=np.float32)
PREFIX = "Represent this sentence for searching relevant passages: "
def embed_query(q): return np.array(next(iter(model.embed([PREFIX + q]))), dtype=np.float32)

def semantic(q, k=3, flt=None):
    sims = vectors @ embed_query(q)
    order = [i for i in np.argsort(-sims) if matches(chunks[i]["meta"], flt)][:k]
    return [(chunks[i], float(sims[i])) for i in order]

WORD = re.compile(r"[a-z0-9]+")
STOP = {"the", "a", "an", "of", "to", "is", "are", "do", "does", "how", "what", "when", "i", "my", "can", "or", "and", "in", "for", "after", "many", "much"}
def keyword(q, k=3, flt=None):
    qw = {w for w in WORD.findall(q.lower()) if w not in STOP}
    scored = [(len(qw & set(WORD.findall(c["text"].lower()))), i) for i, c in enumerate(chunks) if matches(c["meta"], flt)]
    scored = sorted([s for s in scored if s[0]], key=lambda x: (-x[0], x[1]))
    return [(chunks[i], float(n)) for n, i in scored[:k]]

def hybrid(q, k=3, flt=None):
    fused, by_id = {}, {}
    for lst in (semantic(q, max(k * 4, 20), flt), keyword(q, max(k * 4, 20), flt)):
        for rank, (c, _) in enumerate(lst, start=1):
            fused[c["id"]] = fused.get(c["id"], 0.0) + 1.0 / (60 + rank)
            by_id.setdefault(c["id"], c)
    return [(by_id[i], fused[i]) for i in sorted(fused, key=lambda i: -fused[i])[:k]]
print(len(chunks), "chunks ready")

## Step 3: the real RAG and scoring code
This cell writes the project's own `rag.py` logic: `extract_answer` (pick the best sentence and cite it), `retrieval_scores`, `is_correct`, `is_grounded`, `citation_ok`, `failure_label` and `summarise`. Read it. These small functions are the whole "evaluation framework".

In [ ]:
%%writefile rag_core.py
"""Week 2, Day 4: a small RAG pipeline on the Day 3 vector store, and the checks that score it.

Stages (kept separate, so a failure can be placed in one of them):
  1. retrieve   VectorLab.search (semantic or hybrid), optional metadata filter of the signed-in user
  2. gate       optional evidence gate: refuse when the best similarity is below a threshold
  3. generate   EXTRACTIVE answer: the sentence of the retrieved text that shares most words with the question, cited as [n].
                No language model is used here, so this measures retrieval and the answer rule, not a model's writing.
                (A language-model generator would plug in at the same place; it was not run on Day 4.)

Scoring is plain code with no model: retrieval (hit, MRR, context precision, context recall), answer (correct, grounded, citation,
refusal) and a failure label per question.
"""
from __future__ import annotations

import re
import time
from dataclasses import dataclass

from rag_eval_set import RQ

REFUSAL = "I could not find this in the approved documents."
_WORD = re.compile(r"[a-z0-9]+")
_STOP = {"the", "a", "an", "of", "to", "is", "are", "do", "does", "how", "what", "when", "who", "which", "i", "my", "can", "or", "and", "in", "for",
         "after", "many", "much", "be", "it", "on", "at", "by", "me", "you", "we", "with", "from", "as", "that", "this", "get", "long", "often"}
_SENT = re.compile(r"(?<=[.!?])\s+|\n+")


def norm(text: str) -> str:
    return " ".join(_WORD.findall(text.lower()))


def sentences(text: str) -> list[str]:
    return [s.strip() for s in _SENT.split(text) if s.strip()]


def _content_words(text: str) -> set[str]:
    return {w for w in _WORD.findall(text.lower()) if w not in _STOP}


def extract_answer(question: str, hits: list[Hit]) -> tuple[str, int | None]:
    """The best sentence of the retrieved text for this question, and the number [n] of the chunk it came from."""
    qw = _content_words(question)
    best, best_score, best_n = "", 0, None
    for n, h in enumerate(hits, start=1):
        for s in sentences(h.text):
            score = len(qw & _content_words(s))
            if score > best_score:
                best, best_score, best_n = s, score, n
    if best_n is None:
        return REFUSAL, None
    return f"{best} [{best_n}]", best_n


@dataclass
class Hit:
    chunk_id: str
    doc_id: str
    title: str
    text: str
    metadata: dict
    score: float
    rank: int = 0
    sem: float | None = None


@dataclass
class Config:
    name: str
    mode: str = "semantic"
    k: int = 3
    use_filter: bool = False
    gate: float | None = None  # refuse when the best semantic similarity is below this
    rerank_from: int | None = None  # retrieve this many chunks, then let a cross-encoder pick the best k


@dataclass
class Result:
    q: RQ
    answer: str
    cited: int | None
    hits: list[Hit]
    refused: bool
    best_sim: float | None
    retrieve_ms: float = 0.0
    gen_ms: float = 0.0
    prompt_tokens: int = 0
    output_tokens: int = 0


_reranker = None


def rerank(question: str, hits: list[Hit], k: int) -> list[Hit]:
    """A cross-encoder reads the question and each chunk TOGETHER and scores how well the chunk answers it (slower, more precise than vectors)."""
    global _reranker
    if _reranker is None:
        from fastembed.rerank.cross_encoder import TextCrossEncoder
        _reranker = TextCrossEncoder(model_name="Xenova/ms-marco-MiniLM-L-6-v2")
    scores = list(_reranker.rerank(question, [h.text for h in hits]))
    order = sorted(range(len(hits)), key=lambda i: -scores[i])[:k]
    return [Hit(hits[i].chunk_id, hits[i].doc_id, hits[i].title, hits[i].text, hits[i].metadata, round(float(scores[i]), 4), r + 1, hits[i].sem) for r, i in enumerate(order)]


def retrieve(lab: VectorLab, q: RQ, cfg: Config) -> tuple[list[Hit], float | None, float]:
    """Stage 1 (and the gate's input): the chunks, the best semantic similarity, and the time taken in ms."""
    t0 = time.perf_counter()
    flt = q.filter if cfg.use_filter and q.filter else None
    hits, _ = lab.search(q.question, cfg.mode, cfg.rerank_from or cfg.k, flt)
    if cfg.rerank_from:
        hits = rerank(q.question, hits, cfg.k)
    sem, _ = lab.search(q.question, "semantic", 1, flt)
    return hits, (sem[0].sem if sem else None), (time.perf_counter() - t0) * 1000


# ---- scoring (pure functions, no model, no database) ----
def retrieval_scores(q: RQ, hits: list[Hit]) -> dict:
    ids = [h.doc_id for h in hits]
    rank = next((i + 1 for i, d in enumerate(ids) if d in q.docs), None)
    return {"hit": rank is not None, "rr": 1 / rank if rank else 0.0,
            "precision": (sum(d in q.docs for d in ids) / len(ids)) if ids else 0.0,
            "context_recall": any(norm(q.phrase) in norm(h.text) for h in hits) if q.phrase else None}


def is_correct(q: RQ, answer_text: str) -> bool:
    return bool(q.phrase) and norm(q.phrase) in norm(answer_text)


def is_grounded(answer_text: str, hits: list[Hit], cited: int | None) -> bool:
    """Every sentence of the answer (without its [n] mark) appears word for word in the chunk it cites."""
    if cited is None or not 1 <= cited <= len(hits):
        return False
    body = re.sub(r"\s*\[\d+\]\s*$", "", answer_text)
    chunk = norm(hits[cited - 1].text)
    return all(norm(s) in chunk for s in sentences(body))


def citation_ok(q: RQ, hits: list[Hit], cited: int | None) -> bool:
    """The cited chunk belongs to a correct document and holds the expected fact."""
    if cited is None or not 1 <= cited <= len(hits) or not q.phrase:
        return False
    h = hits[cited - 1]
    return h.doc_id in q.docs and norm(q.phrase) in norm(h.text)


def failure_label(q: RQ, r: Result) -> str | None:
    """Where a failed question went wrong. None = passed."""
    if not q.docs:  # not in the documents: the right behaviour is a refusal
        return None if r.refused else "hallucination (answered a question the documents do not cover)"
    if r.refused:
        return "false refusal (the gate or the retrieval returned nothing)"
    rs = retrieval_scores(q, r.hits)
    if not rs["context_recall"]:
        return "retrieval failure (the fact is not in the retrieved text)"
    if not is_correct(q, r.answer):
        return "generation failure (the fact was retrieved but not in the answer)"
    if not citation_ok(q, r.hits, r.cited):
        return "citation failure (right words, wrong source)"
    return None


def summarise(results: list[Result]) -> dict:
    ans = [r for r in results if r.q.docs]
    oos = [r for r in results if not r.q.docs]
    n = len(ans)
    rs = [retrieval_scores(r.q, r.hits) for r in ans]
    return {
        "answerable": n, "not_in_documents": len(oos),
        "hit": round(100 * sum(x["hit"] for x in rs) / n, 1),
        "mrr": round(sum(x["rr"] for x in rs) / n, 3),
        "context_precision": round(100 * sum(x["precision"] for x in rs) / n, 1),
        "context_recall": round(100 * sum(bool(x["context_recall"]) for x in rs) / n, 1),
        "answer_correct": round(100 * sum(is_correct(r.q, r.answer) for r in ans) / n, 1),
        "grounded": round(100 * sum(is_grounded(r.answer, r.hits, r.cited) for r in ans if not r.refused) / max(sum(not r.refused for r in ans), 1), 1),
        "citation_ok": round(100 * sum(citation_ok(r.q, r.hits, r.cited) for r in ans) / n, 1),
        "false_refusals": sum(r.refused for r in ans),
        "refused_correctly": sum(r.refused for r in oos),
        "hallucinated": sum(not r.refused for r in oos),
    }


# ---- extra checks: unsupported numbers, relevance, regression ----
_NUM = re.compile(r"\d[\d,]*(?:\.\d+)?")


def unsupported_numbers(answer_text: str, hits: list[Hit], cited: int | None) -> list[str]:
    """Numbers that appear in the answer but not in the chunk it cites: the cheapest hallucination check for facts like days, amounts and limits."""
    if cited is None or not 1 <= cited <= len(hits):
        return _NUM.findall(re.sub(r"\[\d+\]", "", answer_text))
    context = hits[cited - 1].text
    body = re.sub(r"\[\d+\]", "", answer_text)
    return [n for n in _NUM.findall(body) if n.strip(",") not in context.replace(",", "") and n not in context]


def answer_relevance(lab: VectorLab, question: str, answer_text: str) -> float:
    """How much the answer is about the question (embedding similarity, 0 to 1). A fluent answer to a different question scores low."""
    return lab.similarity(question, re.sub(r"\s*\[\d+\]\s*$", "", answer_text))


def context_relevance(lab: VectorLab, question: str, hits: list[Hit]) -> float:
    """Average similarity between the question and the retrieved chunks: how much of the context is on topic."""
    return round(sum(lab.similarity(question, h.text) for h in hits) / len(hits), 4) if hits else 0.0


def compare_to_baseline(baseline: dict, current: dict, tolerance: float = 2.0) -> list[str]:
    """Regression check: which headline numbers got worse than the saved baseline by more than `tolerance` points."""
    worse = []
    for name, base in baseline.items():
        cur = current.get(name)
        if cur is None:
            worse.append(f"{name}: configuration missing")
            continue
        for key in ("hit", "context_recall", "answer_correct", "citation_ok"):
            if cur[key] < base[key] - tolerance:
                worse.append(f"{name}: {key} fell from {base[key]} to {cur[key]}")
        if cur["mrr"] < base["mrr"] - tolerance / 100:
            worse.append(f"{name}: mrr fell from {base['mrr']} to {cur['mrr']}")
    return worse

In [ ]:
import time
from rag_core import (Hit, Config, Result, REFUSAL, extract_answer, retrieval_scores, is_correct, is_grounded, citation_ok, failure_label, summarise,
                      rerank, unsupported_numbers, answer_relevance, context_relevance, compare_to_baseline)

class Lab:
    # stands in for the project's database object: only the similarity helper is needed for the relevance scores
    def similarity(self, question, text):
        return float(embed_query(question) @ np.array(next(iter(model.embed([text]))), dtype=np.float32))
lab = Lab()

def to_hits(pairs, mode):
    return [Hit(c["id"], c["doc_id"], c["title"], c["text"], c["meta"], round(s, 4), r + 1, s if mode == "semantic" else None) for r, (c, s) in enumerate(pairs)]

def answer_mem(q, cfg):
    '''The same three stages as the real pipeline: retrieve, gate, generate.'''
    t0 = time.perf_counter()
    flt = q.filter if cfg.use_filter and q.filter else None
    pairs = {"semantic": semantic, "hybrid": hybrid, "keyword": keyword}[cfg.mode](q.question, cfg.rerank_from or cfg.k, flt)
    hits = to_hits(pairs, cfg.mode)
    if cfg.rerank_from:   # a cross-encoder reads question and chunk together and keeps the best k
        hits = rerank(q.question, hits, cfg.k)
    sem = semantic(q.question, 1, flt)
    best_sim = sem[0][1] if sem else None
    ms = (time.perf_counter() - t0) * 1000
    if cfg.gate is not None and (best_sim is None or best_sim < cfg.gate):
        return Result(q, REFUSAL, None, hits, True, best_sim, retrieve_ms=ms)
    text, n = extract_answer(q.question, hits)
    return Result(q, text, n, hits, n is None, best_sim, retrieve_ms=ms)

## Step 4: one question, all the way through
Look at the three things a RAG answer is made of: the retrieved chunks, the answer sentence, and the citation number [n].

In [ ]:
q = RAG_QUESTIONS[5]
r = answer_mem(q, Config("demo", "semantic", 3, False, None))
print("QUESTION:", q.question)
for n, h in enumerate(r.hits, 1):
    print(f"  [{n}] {h.doc_id}: {h.text[:90]!r}")
print("ANSWER  :", r.answer)
print("expected:", q.expected)

## Step 5: score retrieval and the answer separately
- **Retrieval**: was the right document found (hit), how high (MRR rank), how much of the context is useful (precision), is the expected fact in the context (recall)?
- **Answer**: does it contain the expected fact (correct)? Is the sentence word for word in the cited chunk (grounded)? Does the citation point to the right source?
If retrieval fails the answer cannot be right. If retrieval worked but the answer is wrong, the fault is in the answer step.

In [ ]:
print(retrieval_scores(q, r.hits))
print("correct  :", is_correct(q, r.answer))
print("grounded :", is_grounded(r.answer, r.hits, r.cited))
print("citation :", citation_ok(q, r.hits, r.cited))
print("failure  :", failure_label(q, r))

# the grounded check really catches an invented sentence
fake = "Passwords must have at least 20 characters [1]"
good = Hit("x:1", "remote-all-2025", "t", "Passwords must have at least 14 characters.", {}, 0.5, 1, 0.5)
print("invented sentence grounded?", is_grounded(fake, [good], 1))

## Step 6: four configurations, one change at a time
- **C0** baseline: semantic search, top 3, no filter, no gate.
- **C1** adds the user's profile filter (region, year).
- **C2** switches to hybrid search.
- **C3** adds an evidence gate: refuse when the best similarity is below 0.60 (chosen before the run, from Day 3).
- **C4** adds a **reranker** to C1: retrieve 10 chunks, then a cross-encoder keeps the best 3 (the first run downloads a small model).

Besides quality, the table shows **answer relevance** and **context relevance** (embedding similarity, 0 to 1) and the **retrieval time** in milliseconds.

In [ ]:
CONFIGS = [Config("C0 baseline", "semantic", 3, False, None), Config("C1 + filter", "semantic", 3, True, None),
           Config("C2 + hybrid", "hybrid", 3, True, None), Config("C3 + gate 0.60", "hybrid", 3, True, 0.60),
           Config("C4 + rerank", "semantic", 3, True, None, rerank_from=10)]
runs, summaries = {}, {}
cols = ("hit", "mrr", "context_precision", "answer_correct", "citation_ok", "refused_correctly", "hallucinated")
print(f"{'config':16}" + "".join(f"{c[:10]:>12}" for c in cols) + f"{'ans_rel':>9}{'ctx_rel':>9}{'ms':>8}")
for cfg in CONFIGS:
    res = [answer_mem(x, cfg) for x in RAG_QUESTIONS]
    runs[cfg.name] = res
    s = summarise(res)
    summaries[cfg.name] = s
    ar = round(sum(answer_relevance(lab, r.q.question, r.answer) for r in res if not r.refused) / max(sum(not r.refused for r in res), 1), 3)
    cr = round(sum(context_relevance(lab, r.q.question, r.hits) for r in res) / len(res), 3)
    ms = round(sorted(r.retrieve_ms for r in res)[len(res) // 2], 1)
    print(f"{cfg.name:16}" + "".join(f"{s[c]:>12}" for c in cols) + f"{ar:>9}{cr:>9}{ms:>8}")

## Step 7: where did the failures happen?
Every failed question gets one label: *retrieval failure*, *generation failure*, *citation failure*, *false refusal* or *hallucination* (answered a question that is not in the documents).
The project's result: the **filter** raised correct answers from 68.8% to 87.5%. Hybrid search did not help, and the gate trades one problem for another. Check whether your table agrees.

In [ ]:
for name in ("C0 baseline", "C1 + filter", "C3 + gate 0.60", "C4 + rerank"):
    print("==", name)
    for r in runs[name]:
        label = failure_label(r.q, r)
        if label:
            print("  -", label.split(" (")[0], "|", r.q.question)

## Step 8: a regression check
After any change (a new prompt, a new model, a new chunk size) you must re-run the same questions and check that nothing got worse. The project saves the headline numbers as a **baseline** and compares. Here we save C1 as the baseline, then compare it with C3 (the gate), which is worse on correct answers, to see the check fail.

In [ ]:
keys = ("hit", "mrr", "context_recall", "answer_correct", "citation_ok")
baseline = {"C1": {k: summaries["C1 + filter"][k] for k in keys}}
same = {"C1": dict(baseline["C1"])}
worse = {"C1": {k: summaries["C3 + gate 0.60"][k] for k in keys}}
print("same numbers  ->", compare_to_baseline(baseline, same) or "passed")
print("gate numbers  ->", compare_to_baseline(baseline, worse) or "passed")

## About the LLM judge and a real language-model answer
In the project, `app/rag_llm.py` has a language-model answer step and an **LLM judge** (a second model that says whether every claim is supported by the context). Both run on a local Ollama model and are tested with a fake model. They need Ollama and enough free memory, so this notebook does not run them. If you run Ollama on your own computer, `python run_rag_eval.py --llm` runs them and reports how often the judge agrees with the plain-code number check. A judge is itself a model and can be wrong, so always compare it with a code check.

## What to try next
1. Change the gate in C3 from 0.60 to 0.55 and to 0.65. How many "not in the documents" questions are refused, and how many real questions are refused by mistake?
2. Add a question of your own to `rag_eval_set.py` (copy one `RQ(...)` line) with its expected fact, and re-run Steps 6 and 7.
3. Look at the question about appealing a performance score. The right chunk is retrieved but the answer is wrong. Why? (Hint: read `extract_answer`.) What would fix it?
4. Write in your own words: why must retrieval and the answer be scored separately?

## Where this is in the real code
- Questions: `vector-lab/app/rag_eval_set.py`. Pipeline and scoring: `vector-lab/app/rag.py`. The run: `vector-lab/run_rag_eval.py`. Tests: `vector-lab/tests/test_rag_eval.py`.
- Evidence page: `docs/evidence/week2-day4.md`.